# HW08-09 — PyTorch MLP: регуляризация и оптимизация обучения

**Датасет:** EMNIST Balanced (Вариант B)

- Часть A (S08): E1-E4 — Dropout, BatchNorm, EarlyStopping
- Часть B (S09): O1-O3 — LR диагностика, SGD+momentum, weight decay

## 2.3.1 Импорты, seed и устройство

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
import torchvision
import torchvision.transforms as transforms
import numpy as np
import matplotlib.pyplot as plt
import csv
import json
import copy
import os

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch version: {torch.__version__}")
print(f"Device: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

PyTorch version: 2.5.1+cu121
Device: cpu


## 2.3.2 Данные и DataLoader

In [2]:
import ssl
ssl._create_default_https_context = ssl._create_unverified_context

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

full_train_dataset = torchvision.datasets.EMNIST(
    root="./data", split="balanced", train=True, download=True, transform=transform
)
test_dataset = torchvision.datasets.EMNIST(
    root="./data", split="balanced", train=False, download=True, transform=transform
)

NUM_CLASSES = len(set(full_train_dataset.targets.numpy()))
print(f"EMNIST Balanced: {NUM_CLASSES} classes")

SUBSET_SIZE = 25000
subset_indices = torch.randperm(len(full_train_dataset), generator=torch.Generator().manual_seed(SEED))[:SUBSET_SIZE]
subset_dataset = torch.utils.data.Subset(full_train_dataset, subset_indices)

train_size = int(0.8 * len(subset_dataset))
val_size = len(subset_dataset) - train_size
train_dataset, val_dataset = random_split(
    subset_dataset, [train_size, val_size],
    generator=torch.Generator().manual_seed(SEED)
)

print(f"Train: {len(train_dataset)}, Val: {len(val_dataset)}, Test: {len(test_dataset)}")

BATCH_SIZE = 256

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

EMNIST Balanced: 47 classes
Train: 90240, Val: 22560, Test: 18800


In [3]:
# Sanity check
x_batch, y_batch = next(iter(train_loader))
print(f"x_batch.shape: {x_batch.shape}")
print(f"y_batch.shape: {y_batch.shape}")
print(f"x range: [{x_batch.min():.2f}, {x_batch.max():.2f}]")
print(f"y unique (batch): {sorted(y_batch.unique().tolist())}")
print(f"Num classes: {NUM_CLASSES}")

x_batch.shape: torch.Size([512, 1, 28, 28])
y_batch.shape: torch.Size([512])
x range: [-1.00, 1.00]
y unique (batch): [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46]
Num classes: 47


## 2.3.3 Модели MLP и цикл обучения

In [4]:
class MLPBase(nn.Module):
    def __init__(self, input_dim=784, hidden_dims=(512, 256, 128), num_classes=47):
        super().__init__()
        layers = [nn.Flatten()]
        prev_dim = input_dim
        for h in hidden_dims:
            layers.append(nn.Linear(prev_dim, h))
            layers.append(nn.ReLU())
            prev_dim = h
        layers.append(nn.Linear(prev_dim, num_classes))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


class MLPDropout(nn.Module):
    def __init__(self, input_dim=784, hidden_dims=(512, 256, 128), num_classes=47, dropout_p=0.3):
        super().__init__()
        layers = [nn.Flatten()]
        prev_dim = input_dim
        for h in hidden_dims:
            layers.append(nn.Linear(prev_dim, h))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout_p))
            prev_dim = h
        layers.append(nn.Linear(prev_dim, num_classes))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


class MLPBatchNorm(nn.Module):
    def __init__(self, input_dim=784, hidden_dims=(512, 256, 128), num_classes=47):
        super().__init__()
        layers = [nn.Flatten()]
        prev_dim = input_dim
        for h in hidden_dims:
            layers.append(nn.Linear(prev_dim, h))
            layers.append(nn.BatchNorm1d(h))
            layers.append(nn.ReLU())
            prev_dim = h
        layers.append(nn.Linear(prev_dim, num_classes))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

In [5]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * x.size(0)
        correct += (logits.argmax(1) == y).sum().item()
        total += x.size(0)
    return running_loss / total, correct / total


def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            loss = criterion(logits, y)
            running_loss += loss.item() * x.size(0)
            correct += (logits.argmax(1) == y).sum().item()
            total += x.size(0)
    return running_loss / total, correct / total

In [6]:
def run_experiment(model, train_loader, val_loader, criterion, optimizer,
                   device, num_epochs, early_stopping_patience=None):
    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
    best_val_acc = 0.0
    best_val_loss = float("inf")
    best_state = None
    patience_counter = 0

    for epoch in range(1, num_epochs + 1):
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
        val_loss, val_acc = evaluate(model, val_loader, criterion, device)

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)

        print(f"  Epoch {epoch:02d} | "
              f"train_loss={train_loss:.4f}  train_acc={train_acc:.4f} | "
              f"val_loss={val_loss:.4f}  val_acc={val_acc:.4f}")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_val_loss = val_loss
            best_state = copy.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1

        if early_stopping_patience and patience_counter >= early_stopping_patience:
            print(f"  EarlyStopping at epoch {epoch} (patience={early_stopping_patience})")
            break

    epochs_trained = len(history["train_loss"])
    return history, best_val_acc, best_val_loss, best_state, epochs_trained

---
## Часть A (S08): регуляризация и переобучение (E1–E4)

### E1 — Baseline (без Dropout, без BatchNorm)

In [7]:
NUM_EPOCHS_A = 15
LR_A = 1e-3
HIDDEN_DIMS = (256, 128)

torch.manual_seed(SEED)
model_e1 = MLPBase(hidden_dims=HIDDEN_DIMS).to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer_e1 = optim.Adam(model_e1.parameters(), lr=LR_A)

print("=== E1: Baseline ===")
hist_e1, best_acc_e1, best_loss_e1, state_e1, epochs_e1 = run_experiment(
    model_e1, train_loader, val_loader, criterion, optimizer_e1, DEVICE, NUM_EPOCHS_A
)
print(f"\nE1 best val_acc: {best_acc_e1:.4f}, best val_loss: {best_loss_e1:.4f}")

=== E1: Baseline ===
  Epoch 01 | train_loss=1.7047  train_acc=0.5396 | val_loss=1.1786  val_acc=0.6606
  Epoch 02 | train_loss=1.0093  train_acc=0.7056 | val_loss=0.9174  val_acc=0.7252


KeyboardInterrupt: 

### E2 — Dropout (p=0.3)

In [ ]:
DROPOUT_P = 0.3

torch.manual_seed(SEED)
model_e2 = MLPDropout(hidden_dims=HIDDEN_DIMS, dropout_p=DROPOUT_P).to(DEVICE)
optimizer_e2 = optim.Adam(model_e2.parameters(), lr=LR_A)

print("=== E2: Dropout ===")
hist_e2, best_acc_e2, best_loss_e2, state_e2, epochs_e2 = run_experiment(
    model_e2, train_loader, val_loader, criterion, optimizer_e2, DEVICE, NUM_EPOCHS_A
)
print(f"\nE2 best val_acc: {best_acc_e2:.4f}, best val_loss: {best_loss_e2:.4f}")

### E3 — BatchNorm

In [ ]:
torch.manual_seed(SEED)
model_e3 = MLPBatchNorm(hidden_dims=HIDDEN_DIMS).to(DEVICE)
optimizer_e3 = optim.Adam(model_e3.parameters(), lr=LR_A)

print("=== E3: BatchNorm ===")
hist_e3, best_acc_e3, best_loss_e3, state_e3, epochs_e3 = run_experiment(
    model_e3, train_loader, val_loader, criterion, optimizer_e3, DEVICE, NUM_EPOCHS_A
)
print(f"\nE3 best val_acc: {best_acc_e3:.4f}, best val_loss: {best_loss_e3:.4f}")

### Сравнение E2 и E3 → выбор лучшего для E4

In [ ]:
print(f"E2 (Dropout)   best val_acc = {best_acc_e2:.4f}")
print(f"E3 (BatchNorm) best val_acc = {best_acc_e3:.4f}")

if best_acc_e2 >= best_acc_e3:
    best_regularizer = "Dropout"
    print(f"\n=> Лучший: E2 (Dropout). Используем его для E4.")
else:
    best_regularizer = "BatchNorm"
    print(f"\n=> Лучший: E3 (BatchNorm). Используем его для E4.")

### E4 — EarlyStopping (лучший из E2/E3 + patience=5)

In [ ]:
PATIENCE = 5
NUM_EPOCHS_E4 = 30

torch.manual_seed(SEED)
if best_regularizer == "Dropout":
    model_e4 = MLPDropout(hidden_dims=HIDDEN_DIMS, dropout_p=DROPOUT_P).to(DEVICE)
    e4_summary = f"[256,128]/ReLU/Dropout(p={DROPOUT_P})"
else:
    model_e4 = MLPBatchNorm(hidden_dims=HIDDEN_DIMS).to(DEVICE)
    e4_summary = "[256,128]/ReLU/BatchNorm"

optimizer_e4 = optim.Adam(model_e4.parameters(), lr=LR_A)

print(f"=== E4: {best_regularizer} + EarlyStopping (patience={PATIENCE}) ===")
hist_e4, best_acc_e4, best_loss_e4, state_e4, epochs_e4 = run_experiment(
    model_e4, train_loader, val_loader, criterion, optimizer_e4,
    DEVICE, NUM_EPOCHS_E4, early_stopping_patience=PATIENCE
)
print(f"\nE4 best val_acc: {best_acc_e4:.4f}, best val_loss: {best_loss_e4:.4f}, epochs: {epochs_e4}")

### Графики части A: train/val loss и accuracy по эпохам (E1–E4)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for label, hist in [("E1 base", hist_e1), ("E2 Dropout", hist_e2),
                     ("E3 BatchNorm", hist_e3), ("E4 EarlyStop", hist_e4)]:
    axes[0].plot(hist["train_loss"], label=f"{label} train", linestyle="--")
    axes[0].plot(hist["val_loss"], label=f"{label} val")

axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].set_title("Loss по эпохам (E1–E4)")
axes[0].legend(fontsize=7)
axes[0].grid(True, alpha=0.3)

for label, hist in [("E1 base", hist_e1), ("E2 Dropout", hist_e2),
                     ("E3 BatchNorm", hist_e3), ("E4 EarlyStop", hist_e4)]:
    axes[1].plot(hist["train_acc"], label=f"{label} train", linestyle="--")
    axes[1].plot(hist["val_acc"], label=f"{label} val")

axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].set_title("Accuracy по эпохам (E1–E4)")
axes[1].legend(fontsize=7)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### График лучшего прогона E4 → `curves_best.png`

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

epochs_range = range(1, epochs_e4 + 1)
axes[0].plot(epochs_range, hist_e4["train_loss"], label="train_loss")
axes[0].plot(epochs_range, hist_e4["val_loss"], label="val_loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].set_title("E4: Loss")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs_range, hist_e4["train_acc"], label="train_acc")
axes[1].plot(epochs_range, hist_e4["val_acc"], label="val_acc")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].set_title("E4: Accuracy")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
os.makedirs("artifacts/figures", exist_ok=True)
fig.savefig("artifacts/figures/curves_best.png", dpi=150, bbox_inches="tight")
print("Saved: artifacts/figures/curves_best.png")
plt.show()

---
## Часть B (S09): LR, оптимизаторы, weight decay (O1–O3)

### O1 — LR слишком большой (Adam, lr=0.1)

In [ ]:
NUM_EPOCHS_O = 6

torch.manual_seed(SEED)
if best_regularizer == "Dropout":
    model_o1 = MLPDropout(hidden_dims=HIDDEN_DIMS, dropout_p=DROPOUT_P).to(DEVICE)
else:
    model_o1 = MLPBatchNorm(hidden_dims=HIDDEN_DIMS).to(DEVICE)

optimizer_o1 = optim.Adam(model_o1.parameters(), lr=1e-1)

print("=== O1: LR слишком большой (lr=0.1) ===")
hist_o1, best_acc_o1, best_loss_o1, _, epochs_o1 = run_experiment(
    model_o1, train_loader, val_loader, criterion, optimizer_o1, DEVICE, NUM_EPOCHS_O
)
print(f"\nO1 best val_acc: {best_acc_o1:.4f}")

### O2 — LR слишком маленький (Adam, lr=1e-5)

In [ ]:
torch.manual_seed(SEED)
if best_regularizer == "Dropout":
    model_o2 = MLPDropout(hidden_dims=HIDDEN_DIMS, dropout_p=DROPOUT_P).to(DEVICE)
else:
    model_o2 = MLPBatchNorm(hidden_dims=HIDDEN_DIMS).to(DEVICE)

optimizer_o2 = optim.Adam(model_o2.parameters(), lr=1e-5)

print("=== O2: LR слишком маленький (lr=1e-5) ===")
hist_o2, best_acc_o2, best_loss_o2, _, epochs_o2 = run_experiment(
    model_o2, train_loader, val_loader, criterion, optimizer_o2, DEVICE, NUM_EPOCHS_O
)
print(f"\nO2 best val_acc: {best_acc_o2:.4f}")

### O3 — SGD + momentum + weight decay

In [ ]:
NUM_EPOCHS_O3 = 10

torch.manual_seed(SEED)
if best_regularizer == "Dropout":
    model_o3 = MLPDropout(hidden_dims=HIDDEN_DIMS, dropout_p=DROPOUT_P).to(DEVICE)
else:
    model_o3 = MLPBatchNorm(hidden_dims=HIDDEN_DIMS).to(DEVICE)

optimizer_o3 = optim.SGD(model_o3.parameters(), lr=1e-2, momentum=0.9, weight_decay=1e-4)

print("=== O3: SGD + momentum=0.9 + weight_decay=1e-4 (lr=0.01) ===")
hist_o3, best_acc_o3, best_loss_o3, _, epochs_o3 = run_experiment(
    model_o3, train_loader, val_loader, criterion, optimizer_o3, DEVICE, NUM_EPOCHS_O3
)
print(f"\nO3 best val_acc: {best_acc_o3:.4f}")

### График LR extremes (O1 и O2) → `curves_lr_extremes.png`

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

axes[0, 0].plot(hist_o1["train_loss"], label="train", marker="o")
axes[0, 0].plot(hist_o1["val_loss"], label="val", marker="s")
axes[0, 0].set_title("O1 (lr=0.1): Loss")
axes[0, 0].set_xlabel("Epoch")
axes[0, 0].set_ylabel("Loss")
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

axes[0, 1].plot(hist_o1["train_acc"], label="train", marker="o")
axes[0, 1].plot(hist_o1["val_acc"], label="val", marker="s")
axes[0, 1].set_title("O1 (lr=0.1): Accuracy")
axes[0, 1].set_xlabel("Epoch")
axes[0, 1].set_ylabel("Accuracy")
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

axes[1, 0].plot(hist_o2["train_loss"], label="train", marker="o")
axes[1, 0].plot(hist_o2["val_loss"], label="val", marker="s")
axes[1, 0].set_title("O2 (lr=1e-5): Loss")
axes[1, 0].set_xlabel("Epoch")
axes[1, 0].set_ylabel("Loss")
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

axes[1, 1].plot(hist_o2["train_acc"], label="train", marker="o")
axes[1, 1].plot(hist_o2["val_acc"], label="val", marker="s")
axes[1, 1].set_title("O2 (lr=1e-5): Accuracy")
axes[1, 1].set_xlabel("Epoch")
axes[1, 1].set_ylabel("Accuracy")
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
fig.savefig("artifacts/figures/curves_lr_extremes.png", dpi=150, bbox_inches="tight")
print("Saved: artifacts/figures/curves_lr_extremes.png")
plt.show()

---
## Финальная оценка лучшей модели (E4) на test

In [ ]:
model_e4.load_state_dict(state_e4)
test_loss, test_acc = evaluate(model_e4, test_loader, criterion, DEVICE)
print(f"\n=== Финальная оценка на TEST ===")
print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_acc:.4f}")

---
## Сохранение артефактов

In [ ]:
# best_model.pt
torch.save(state_e4, "artifacts/best_model.pt")
print("Saved: artifacts/best_model.pt")

# best_config.json
best_config = {
    "dataset": "EMNIST_balanced",
    "seed": SEED,
    "input_dim": 784,
    "hidden_dims": list(HIDDEN_DIMS),
    "num_classes": NUM_CLASSES,
    "activation": "ReLU",
    "regularization": best_regularizer,
    "dropout_p": DROPOUT_P if best_regularizer == "Dropout" else None,
    "optimizer": "Adam",
    "lr": LR_A,
    "batch_size": BATCH_SIZE,
    "early_stopping_patience": PATIENCE,
    "epochs_trained": epochs_e4,
    "best_val_accuracy": round(best_acc_e4, 4),
    "test_accuracy": round(test_acc, 4)
}
with open("artifacts/best_config.json", "w") as f:
    json.dump(best_config, f, indent=2, ensure_ascii=False)
print("Saved: artifacts/best_config.json")

In [ ]:
# runs.csv
runs = [
    {
        "experiment_id": "E1",
        "dataset": "EMNIST_balanced",
        "seed": SEED,
        "model_summary": "[256,128]/ReLU/no_reg",
        "optimizer": "Adam",
        "lr": LR_A,
        "momentum": 0,
        "weight_decay": 0,
        "epochs_trained": epochs_e1,
        "best_val_accuracy": round(best_acc_e1, 4),
        "best_val_loss": round(best_loss_e1, 4),
    },
    {
        "experiment_id": "E2",
        "dataset": "EMNIST_balanced",
        "seed": SEED,
        "model_summary": f"[256,128]/ReLU/Dropout(p={DROPOUT_P})",
        "optimizer": "Adam",
        "lr": LR_A,
        "momentum": 0,
        "weight_decay": 0,
        "epochs_trained": epochs_e2,
        "best_val_accuracy": round(best_acc_e2, 4),
        "best_val_loss": round(best_loss_e2, 4),
    },
    {
        "experiment_id": "E3",
        "dataset": "EMNIST_balanced",
        "seed": SEED,
        "model_summary": "[256,128]/ReLU/BatchNorm",
        "optimizer": "Adam",
        "lr": LR_A,
        "momentum": 0,
        "weight_decay": 0,
        "epochs_trained": epochs_e3,
        "best_val_accuracy": round(best_acc_e3, 4),
        "best_val_loss": round(best_loss_e3, 4),
    },
    {
        "experiment_id": "E4",
        "dataset": "EMNIST_balanced",
        "seed": SEED,
        "model_summary": e4_summary,
        "optimizer": "Adam",
        "lr": LR_A,
        "momentum": 0,
        "weight_decay": 0,
        "epochs_trained": epochs_e4,
        "best_val_accuracy": round(best_acc_e4, 4),
        "best_val_loss": round(best_loss_e4, 4),
    },
    {
        "experiment_id": "O1",
        "dataset": "EMNIST_balanced",
        "seed": SEED,
        "model_summary": e4_summary,
        "optimizer": "Adam",
        "lr": 0.1,
        "momentum": 0,
        "weight_decay": 0,
        "epochs_trained": epochs_o1,
        "best_val_accuracy": round(best_acc_o1, 4),
        "best_val_loss": round(best_loss_o1, 4),
    },
    {
        "experiment_id": "O2",
        "dataset": "EMNIST_balanced",
        "seed": SEED,
        "model_summary": e4_summary,
        "optimizer": "Adam",
        "lr": 1e-5,
        "momentum": 0,
        "weight_decay": 0,
        "epochs_trained": epochs_o2,
        "best_val_accuracy": round(best_acc_o2, 4),
        "best_val_loss": round(best_loss_o2, 4),
    },
    {
        "experiment_id": "O3",
        "dataset": "EMNIST_balanced",
        "seed": SEED,
        "model_summary": e4_summary,
        "optimizer": "SGD",
        "lr": 0.01,
        "momentum": 0.9,
        "weight_decay": 1e-4,
        "epochs_trained": epochs_o3,
        "best_val_accuracy": round(best_acc_o3, 4),
        "best_val_loss": round(best_loss_o3, 4),
    },
]

fieldnames = list(runs[0].keys())
with open("artifacts/runs.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(runs)
print("Saved: artifacts/runs.csv")

print("\n=== Сводка ===")
print(f"{'ID':<5} {'val_acc':>9} {'val_loss':>9} {'epochs':>7}")
print("-" * 35)
for r in runs:
    print(f"{r['experiment_id']:<5} {r['best_val_accuracy']:>9.4f} {r['best_val_loss']:>9.4f} {r['epochs_trained']:>7}")

In [ ]:
print("\n=== Все артефакты ===")
for root, dirs, files in os.walk("artifacts"):
    for f in files:
        path = os.path.join(root, f)
        size = os.path.getsize(path)
        print(f"  {path}  ({size:,} bytes)")